# Train Chess Figurine Classifier (HOG + MLP → TFLite)

Train a 5-class classifier to recognize chess piece figurines (K, Q, R, B, N)
using **HOG (Histogram of Oriented Gradients) features** fed into a small **Keras MLP**,
exported to TFLite.

**Why HOG instead of raw pixels?**
HOG captures gradient structure (edges, shapes) that distinguishes K/Q/R/B/N regardless
of font, DPI, or rendering style — much more robust than pixel-level CNN features.

**Why a custom HOG (not scikit-image)?**
The HOG computation mirrors `HogExtractor.extract()` in Flutter **exactly** —
no rounding differences, no library version drift.

**Why ±35° rotation augmentation?**
PDF pages are sometimes scanned at a slight angle (up to ~30°). Augmenting with ±35°
makes the model robust to tilted glyphs.

**Input:** `chess_glyphs_classifier.zip` from `extract_chess_glyphs`, uploaded to Drive.  
**Output:** `figurine_classifier.tflite` — model input is a 1767-float HOG vector.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

## Step 2 — Copy zip from Drive and extract locally

Upload `chess_glyphs_classifier.zip` (produced by `extract_chess_glyphs`) to Google Drive
**as a single zip file** — do not unzip it on your machine first.  
Set `ZIP_ON_DRIVE` to its path, then run this cell.  
The zip is copied to Colab's local disk and extracted there for fast I/O during training.

In [ ]:
import os, shutil, zipfile

# ── EDIT THIS: path to the zip on your Drive ──────────────────────────────
ZIP_ON_DRIVE = '/content/gdrive/MyDrive/entrainement_ocr_echecs/chess_glyphs_classifier.zip'

LOCAL_ZIP    = '/content/chess_glyphs_classifier.zip'
EXTRACT_DIR  = '/content/glyphs_extracted'

if not os.path.exists(ZIP_ON_DRIVE):
    raise FileNotFoundError(
        f'Zip not found on Drive: {ZIP_ON_DRIVE}\n'
        f'Upload chess_glyphs_classifier.zip to that location first.'
    )

# Copy zip to local disk (fast — stays on Colab's own storage)
print('Copying zip from Drive to local disk...')
shutil.copy2(ZIP_ON_DRIVE, LOCAL_ZIP)
size_mb = os.path.getsize(LOCAL_ZIP) / (1024 * 1024)
print(f'  Copied: {size_mb:.1f} MB')

# Extract
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
print('Extracting...')
with zipfile.ZipFile(LOCAL_ZIP, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Locate glyphs/ folder — search in root and one level down
GLYPHS_DIR = None
for candidate in [
    os.path.join(EXTRACT_DIR, 'glyphs'),
    os.path.join(EXTRACT_DIR, 'glyphs_results', 'glyphs'),
]:
    if os.path.exists(candidate):
        GLYPHS_DIR = candidate
        break

if GLYPHS_DIR is None:
    raise FileNotFoundError(
        f'Could not locate glyphs/ folder inside the zip.\n'
        f'Top-level contents: {os.listdir(EXTRACT_DIR)}'
    )

print(f'\n✅ Glyphs extracted to {GLYPHS_DIR}')
from PIL import Image

def count_images(folder):
    count = 0
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        if not os.path.isfile(path):
            continue
        try:
            Image.open(path).verify()
            count += 1
        except Exception:
            pass
    return count

total = 0
for piece in sorted(os.listdir(GLYPHS_DIR)):
    piece_dir = os.path.join(GLYPHS_DIR, piece)
    if not os.path.isdir(piece_dir):
        continue
    n = count_images(piece_dir)
    total += n
    print(f'  {"✅" if n > 0 else "❌"} {piece}: {n} images')
print(f'  Total: {total} images')

## Step 3 — Install dependencies

In [ ]:
!pip install -q tensorflow pillow numpy scikit-image scikit-learn matplotlib

## Step 4 — Configuration and HOG feature extractor

**`_compute_hog_features` is the reference implementation.**  
It mirrors `HogExtractor.extract()` in `lib/chess/hog_extractor.dart` line-for-line.  
Do **not** replace this with `skimage.feature.hog` — scikit-image's internals differ
subtly and would break training/inference consistency.

In [ ]:
import numpy as np
from PIL import Image, ImageOps, ImageFilter
from skimage import transform
import random

CLASS_NAMES       = ['K', 'Q', 'R', 'B', 'N']
IMG_SIZE          = 32
AUGMENT_PER_IMAGE = 40      # augmented variants per original image
MAX_ROTATION_DEG  = 35      # ±35° covers scanned pages tilted up to ~30°
EPOCHS            = 80
BATCH_SIZE        = 128
VAL_SPLIT         = 0.15
TFLITE_PATH       = 'figurine_classifier.tflite'

# HOG constants — must stay in sync with hog_extractor.dart
_ORIENTATIONS = 9
_PX_PER_CELL  = 4
_CPB          = 2
_N_CELLS      = IMG_SIZE // _PX_PER_CELL          # 8
_N_BLOCKS     = _N_CELLS - _CPB + 1               # 7
_BLOCK_SIZE   = _CPB * _CPB * _ORIENTATIONS       # 36
FEATURE_DIM   = _N_BLOCKS * _N_BLOCKS * _BLOCK_SIZE + 3  # 1767


def _compute_hog_features(img_32x32: np.ndarray) -> np.ndarray:
    """
    Custom HOG matching Dart HogExtractor.extract() exactly.
    img_32x32: float64 array (32, 32), values in [0, 1].
    Returns float64 array of length 1764.
    """
    H, W = IMG_SIZE, IMG_SIZE

    # 1. Gradients — central difference, zero at boundary
    gx = np.zeros((H, W), dtype=np.float64)
    gy = np.zeros((H, W), dtype=np.float64)
    gx[:, 1:-1] = img_32x32[:, 2:] - img_32x32[:, :-2]
    gy[1:-1, :] = img_32x32[2:, :] - img_32x32[:-2, :]

    # 2. Magnitude and unsigned angle [0°, 180°)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    ang = np.degrees(np.arctan2(gy, gx)) % 180.0

    # 3. Cell histograms
    bin_width  = 180.0 / _ORIENTATIONS
    cell_hists = np.zeros((_N_CELLS, _N_CELLS, _ORIENTATIONS), dtype=np.float64)

    for y in range(H):
        cy = y // _PX_PER_CELL
        for x in range(W):
            cx  = x // _PX_PER_CELL
            m   = mag[y, x]
            bf  = ang[y, x] / bin_width
            b0  = int(bf) % _ORIENTATIONS
            b1  = (b0 + 1) % _ORIENTATIONS
            t   = bf - int(bf)
            cell_hists[cy, cx, b0] += m * (1.0 - t)
            cell_hists[cy, cx, b1] += m * t

    # 4. Block normalization (L2-Hys)
    eps2      = 1e-5 ** 2
    hog_feats = []
    for by in range(_N_BLOCKS):
        for bx in range(_N_BLOCKS):
            block = cell_hists[by:by + _CPB, bx:bx + _CPB, :].ravel().copy()
            block /= np.sqrt(np.dot(block, block) + eps2)
            np.clip(block, 0.0, 0.2, out=block)
            block /= np.sqrt(np.dot(block, block) + eps2)
            hog_feats.append(block)

    return np.concatenate(hog_feats)  # length 1764


def extract_image_features(img) -> np.ndarray:
    """HOG + shape/intensity stats. img: PIL Image. Returns float32[1767]."""
    if img.width < 5 or img.height < 5:
        return None
    img_gray    = np.array(img.convert('L'), dtype=np.float64) / 255.0
    img_resized = transform.resize(img_gray, (IMG_SIZE, IMG_SIZE), anti_aliasing=True)
    hog_feat    = _compute_hog_features(img_resized)
    extra = np.array([
        img.width / max(img.height, 1),
        np.mean(img_resized),
        np.std(img_resized),
    ], dtype=np.float64)
    return np.concatenate([hog_feat, extra]).astype(np.float32)


sample = extract_image_features(Image.new('L', (IMG_SIZE, IMG_SIZE), 128))
assert len(sample) == FEATURE_DIM
print(f'HOG feature vector : {FEATURE_DIM} dims  ✅')
print(f'Classes            : {CLASS_NAMES}')
print(f'Max rotation       : ±{MAX_ROTATION_DEG}° (robust to ~30° page tilt)')
print(f'Augmentation       : {AUGMENT_PER_IMAGE} variants per image')
print(f'Output             : {TFLITE_PATH}')

## Step 5 — Load labeled glyph images

In [ ]:
from collections import defaultdict

def load_glyphs(glyphs_dir, class_names):
    images, labels, counts = [], [], defaultdict(int)
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(glyphs_dir, class_name)
        if not os.path.exists(class_dir):
            print(f'⚠️  {class_name} folder not found')
            continue
        for filename in sorted(os.listdir(class_dir)):
            filepath = os.path.join(class_dir, filename)
            if not os.path.isfile(filepath):
                continue
            try:
                img = Image.open(filepath).convert('L')
                images.append(img)
                labels.append(class_idx)
                counts[class_name] += 1
            except Exception:
                pass
    return images, labels, counts

print(f'Loading glyphs from {GLYPHS_DIR}...\n')
images, labels, counts = load_glyphs(GLYPHS_DIR, CLASS_NAMES)
print(f'✅ Loaded {len(images)} glyph images\n')
for name in CLASS_NAMES:
    n = counts[name]
    print(f'  {"✅" if n > 0 else "❌"} {name}: {n}')

## Step 6 — Data augmentation + HOG feature extraction

In [ ]:
def augment_image(img, n, size=32):
    base = img.convert('L').resize((size, size), Image.LANCZOS)
    arr  = np.array(base, dtype=np.float32) / 255.0
    results = []
    for _ in range(n):
        pil   = Image.fromarray((arr * 255).astype(np.uint8))
        angle = random.uniform(-MAX_ROTATION_DEG, MAX_ROTATION_DEG)
        pil   = pil.rotate(angle, resample=Image.BICUBIC, fillcolor=255)

        scale    = random.uniform(0.80, 1.20)
        new_size = max(4, int(size * scale))
        pil      = pil.resize((new_size, new_size), Image.LANCZOS)

        canvas = Image.new('L', (size, size), 255)
        offset = (size - new_size) // 2
        canvas.paste(pil, (offset, offset))

        if random.random() > 0.5:
            canvas = ImageOps.mirror(canvas)
        if random.random() > 0.8:
            canvas = canvas.filter(
                ImageFilter.GaussianBlur(radius=random.uniform(0.2, 0.6)))

        aug_arr = np.array(canvas, dtype=np.float32) / 255.0
        aug_arr += np.random.normal(0, 0.03, aug_arr.shape)
        results.append(Image.fromarray(
            (np.clip(aug_arr, 0.0, 1.0) * 255).astype(np.uint8)))
    return results


print(f'Augmenting + extracting HOG features '
      f'({AUGMENT_PER_IMAGE} variants × {len(images)} images, '
      f'rotation ±{MAX_ROTATION_DEG}°)...')

X, y = [], []
for img, label in zip(images, labels):
    for aug in augment_image(img, AUGMENT_PER_IMAGE):
        feat = extract_image_features(aug)
        if feat is not None:
            X.append(feat)
            y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y)
print(f'\n✅ {len(X)} augmented samples — feature matrix: {X.shape}')

## Step 7 — Train MLP on HOG features

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=42
)
print(f'Train: {len(X_train)}  Val: {len(X_val)}')

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(FEATURE_DIM,)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
], name='figurine_hog_mlp')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=12, restore_best_weights=True, monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=6, min_lr=1e-6, monitor='val_accuracy'),
    ],
    verbose=1,
)
print(f'\n✅ Best val accuracy: {max(history.history["val_accuracy"]):.4%}')

## Step 8 — Per-class accuracy report

In [ ]:
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

y_pred_probs = model.predict(X_val, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
print(classification_report(y_val, y_pred, target_names=CLASS_NAMES))

print('Per-class spot-check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = y_val == cls_idx
    if mask.sum() == 0:
        continue
    probs = y_pred_probs[mask][0]
    pred  = CLASS_NAMES[np.argmax(probs)]
    print(f'  {"✅" if pred == cls_name else "⚠️ "} {cls_name} → {pred} ({probs.max():.4%})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy');  ax1.legend()
ax2.plot(history.history['loss'],         label='train')
ax2.plot(history.history['val_loss'],     label='val')
ax2.set_title('Loss');      ax2.legend()
plt.tight_layout(); plt.show()

## Step 9 — Export TFLite model

In [ ]:
converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize(TFLITE_PATH) / 1024
print(f'✅ Saved: {TFLITE_PATH} ({size_kb:.0f} KB)')

interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print(f'   Input : {inp["shape"]}  dtype={inp["dtype"].__name__}')
print(f'   Output: {out["shape"]}  dtype={out["dtype"].__name__}')

print('\nTFLite spot-check:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = y_val == cls_idx
    if mask.sum() == 0:
        continue
    sample = X_val[mask][0:1].astype(np.float32)
    interp.set_tensor(inp['index'], sample)
    interp.invoke()
    probs = interp.get_tensor(out['index'])[0]
    pred  = CLASS_NAMES[np.argmax(probs)]
    print(f'  {"✅" if pred == cls_name else "⚠️ "} {cls_name} → {pred} ({probs.max():.4%})')

## Step 10 — Download model

In [ ]:
from google.colab import files
files.download(TFLITE_PATH)
print(f'✅ Downloaded {TFLITE_PATH}')